In [ ]:
import cudf
import dask_cudf
import pickle

import pandas as pd
import gutils
from cuml.dask.ensemble import RandomForestClassifier as cumlDaskRF
from sklearn.linear_model import LogisticRegression

In [ ]:
# Load the cuML model using pickle
with open("../meter.pkl", "rb") as f:
    model: cumlDaskRF
    model = pickle.load(f)

df = pd.read_parquet("../../data/time_slices.parquet")
games = pd.read_parquet("../data/games.parquet")

In [ ]:
game = 30227
season = 2009
filtered_df = df[(df["game"] == game) & (df["season"] == season)]

X = filtered_df.drop(columns=["winner", "game", "season"], axis=1)
X = X.astype("float32")

# convert the pandas DataFrame to CuDF DataFrame
X_cudf = cudf.DataFrame.from_pandas(X)

probabilities = model.predict_proba(dask_cudf.from_cudf(X_cudf, npartitions=1))

probabilities_df = probabilities.compute().to_pandas()

In [ ]:
from graphing import gutils

filtered_game = games[(games["Game_Id"] == str(game)) & (games["Season"] == season)]
home = gutils.team_name_color(filtered_game.iloc[0]["Home_Team"])
away = gutils.team_name_color(filtered_game.iloc[0]["Away_Team"])

gutils.graph_probabilities(filtered_df["time_remaining"] * 3600, probabilities_df.iloc[:, 0], home, away)